# 01 — Train, validate, and test HIVRoBERTa (from the HIVEmotions HF repo)
Reproduces the paper's training pipeline end to end: DAPT → GoEmotions phase → merged HIV phase → threshold tuning (deployed F1 set + β=2 recall set) → evaluation on the balanced 800-message test split. Data comes exclusively from the HIVEmotions dataset repo; credentials come from `config.py` (environment variables take precedence).

**Faithfulness note:** the paper's DAPT stage used 935 messages including four legacy emotion categories that are not part of the released dataset; here DAPT uses the released 595 authored train+validation messages. DAPT is a light adaptation step — supervised training below is identical to the paper — but any retrain lands *near*, not *on*, the reported numbers (GPU nondeterminism alone guarantees that).

In [ ]:
%%capture
!pip install -q "transformers==4.57.1" datasets accelerate safetensors huggingface_hub scikit-learn openai

In [ ]:
import torch
from transformers import set_seed
set_seed(42)
assert torch.cuda.is_available(), "GPU runtime required"
print(torch.cuda.get_device_name(0))

## Data — pull HIVEmotions and verify the published composition

In [ ]:
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict
import config
raw = load_dataset(config.HF_DATASET_REPO, token=config.get_hf_token())
TARGET_EMOTIONS = ["anger","disgust","embarrassment","fear","grief","nervousness","sadness"]
def vec(lab): return [1.0 if e in lab.split(",") else 0.0 for e in TARGET_EMOTIONS]
def sub(split, source): return raw[split].filter(lambda x: x["source"]==source)
counts={s:len(raw[s]) for s in raw}
assert counts=={"train":5041,"validation":741,"test":800,"goemotions_test":595}, counts
go = {"train":sub("train","goemotions"), "validation":sub("validation","goemotions"), "test":raw["goemotions_test"]}
assert [len(go[s]) for s in ["train","validation","test"]]==[4621,566,595]
authored = {s: sub(s,"authored") for s in ["train","validation"]}
assert [len(authored[s]) for s in ["train","validation"]]==[420,175]
print("HIVEmotions loaded and verified:", counts)

## Stage 1 — Domain-adaptive pretraining

In [ ]:
import os
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

model_name = "roberta-large"
texts = list(authored["train"]["text"]) + list(authored["validation"]["text"])
ds = Dataset.from_dict({"text": texts})
print(f"DAPT corpus: {len(ds)} authored messages (released subset — see note above)")

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
        padding="max_length"
    )

print("🔤 Tokenizing for MLM...")
ds_tok = ds.map(tokenize_fn, batched=True, remove_columns=["text"])

collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

print("🔥 Loading MLM model...")
mlm_model = AutoModelForMaskedLM.from_pretrained(model_name)

training_args = TrainingArguments(
    output_dir=save_dir,
    overwrite_output_dir=True,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=200,
    save_strategy="epoch",
    save_total_limit=1,
    prediction_loss_only=True,
    report_to="none",
)

trainer = Trainer(
    model=mlm_model,
    args=training_args,
    train_dataset=ds_tok,
    data_collator=collator,
)

print("🚀 Running DAPT (MLM on HIV train+val text)...")
trainer.train()

print(f"💾 Saving DAPT model to {save_dir}")
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)


## Stage 2 — Two-phase supervised training (also tunes and saves the β=2 recall thresholds)

In [ ]:
#!/usr/bin/env python3
import os
import json
import random
from dataclasses import asdict, dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader
from transformers import (
    AutoConfig,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from transformers import RobertaForSequenceClassification, TrainerCallback
from datasets import load_dataset, concatenate_datasets, Dataset
import pandas as pd
from sklearn.metrics import f1_score, precision_recall_fscore_support

# ============================================================
# 0. CONFIG
# ============================================================

@dataclass
class TrainConfig:
    # Data paths
    goemotions_name: str = "google-research-datasets/go_emotions"
    goemotions_config: str = "simplified"
    # hiv_csv_path removed — data comes from the HIVEmotions HF repo  # ← UPDATE IF NEEDED

    # Save dir (final model + phase2 thresholds saved here)
    save_dir: str = "models/m1_7emotion_roberta_large_dapt"

    # Training
    max_length: int = 384
    batch_size_train: int = 16
    batch_size_eval: int = 32
    num_epochs_goemotions: int = 4
    num_epochs_hiv: int = 8
    lr_goemotions: float = 2e-5
    lr_hiv: float = 1e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1

    # Loss
    focal_gamma: float = 2.0
    label_smoothing: float = 0.1
    synthetic_weight_factor: float = 0.5

    # Early stopping
    patience: int = 3
    seed: int = 42
    gpu_id: int = 0

cfg = TrainConfig()
os.makedirs(cfg.save_dir, exist_ok=True)

# ============================================================
# 1. SEED & DEVICE
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
device = torch.device(f"cuda:{cfg.gpu_id}" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ============================================================
# 2. LOAD & PREPARE GOEMOTIONS (7 emotions only)
# ============================================================

print("📥 Loading GoEmotions (simplified)...")
TARGET_EMOTIONS = ["anger","disgust","embarrassment","fear","grief","nervousness","sadness"]
print("📥 Building GoEmotions component from the HIVEmotions repo...")
ds_go_7 = DatasetDict({s: Dataset.from_dict({
    "text":   go[s]["text"],
    "labels": [vec(l) for l in go[s]["labels"]],
}) for s in ["train","validation","test"]})
print("📊 GoEmotions 7:", {s: len(ds_go_7[s]) for s in ds_go_7})

print(f"📊 GoEmotions 7: train={len(ds_go_7['train'])}, val={len(ds_go_7['validation'])}, test={len(ds_go_7['test'])}")

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=cfg.max_length,
    )

print("🔤 Tokenizing GoEmotions...")
ds_go_tok = ds_go_7.map(tokenize_fn, batched=True, load_from_cache_file=False)

def cast_labels_to_float(batch):
    batch["labels"] = [[float(x) for x in lab] for lab in batch["labels"]]
    return batch

ds_go_tok = ds_go_tok.map(cast_labels_to_float, batched=True, load_from_cache_file=False)

# ============================================================
# 3. PHASE 1: TRAIN ON GOEMOTIONS (NO DISK CHECKPOINTS) + CAPTURE BEST THRESHOLDS
# ============================================================

print("\n🚀 Phase 1: Training RoBERTa on 7 emotions (GoEmotions)...")

dapt_dir = "models/roberta_large_dapt_hiv"
config = AutoConfig.from_pretrained(
    dapt_dir,
    num_labels=7,
    problem_type="multi_label_classification",
)

model = RobertaForSequenceClassification.from_pretrained(dapt_dir, config=config)
model.to(device)

class FloatLabelsCollator:
    def __init__(self, tok):
        self.collator = DataCollatorWithPadding(tok)

    def __call__(self, features):
        batch = self.collator(features)
        if "labels" in batch:
            batch["labels"] = batch["labels"].float()
        return batch

collator = FloatLabelsCollator(tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    macro = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"macro_precision": macro[0], "macro_recall": macro[1], "macro_f1": macro[2]}

def tune_thresholds_per_label(probs, labels, beta=1.0):
    thrs = np.linspace(0.05, 0.95, 19)
    best = np.full(labels.shape[1], 0.5, dtype=np.float32)

    for i in range(labels.shape[1]):
        y = labels[:, i]
        p = probs[:, i]
        best_score = -1.0
        best_t = 0.5

        for t in thrs:
            pred = (p >= t).astype(int)
            tp = ((pred == 1) & (y == 1)).sum()
            fp = ((pred == 1) & (y == 0)).sum()
            fn = ((pred == 0) & (y == 1)).sum()

            if tp + fp == 0 or tp + fn == 0:
                score = 0.0
            else:
                prec = tp / (tp + fp + 1e-8)
                rec = tp / (tp + fn + 1e-8)
                score = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + 1e-8)

            if score > best_score:
                best_score = score
                best_t = float(t)

        best[i] = best_t

    return best

class BestEpochThresholdsCallback(TrainerCallback):
    def __init__(self, trainer_ref, val_ds, label_names):
        self.trainer_ref = trainer_ref
        self.val_ds = val_ds
        self.label_names = label_names
        self.best_macro_f1 = -1.0
        self.best_epoch = None
        self.best_thr_go = None
        self.best_sanity = None

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics is None:
            return
        current = metrics.get("eval_macro_f1", None)
        if current is None:
            return

        if current > self.best_macro_f1 + 1e-12:
            self.best_macro_f1 = float(current)
            self.best_epoch = int(state.epoch) if state.epoch is not None else None

            raw = self.trainer_ref.predict(self.val_ds)
            logits = raw.predictions
            labels_val = raw.label_ids
            probs_val = 1 / (1 + np.exp(-logits))

            thr = tune_thresholds_per_label(probs_val, labels_val, beta=1.0)
            preds_tuned = (probs_val >= thr[None, :]).astype(int)

            prec, rec, f1, _ = precision_recall_fscore_support(
                labels_val, preds_tuned, average=None, zero_division=0
            )
            macro = precision_recall_fscore_support(labels_val, preds_tuned, average="macro", zero_division=0)
            micro = precision_recall_fscore_support(labels_val, preds_tuned, average="micro", zero_division=0)

            self.best_thr_go = thr
            self.best_sanity = {
                "per_class": {
                    name: {"p": float(prec[i]), "r": float(rec[i]), "f1": float(f1[i])}
                    for i, name in enumerate(self.label_names)
                },
                "macro_f1_tuned": float(macro[2]),
                "micro_f1_tuned": float(micro[2]),
            }

            print("\n📌 New BEST epoch (by eval_macro_f1 @0.5):", self.best_epoch, "| eval_macro_f1:", self.best_macro_f1)
            print("✅ Stored tuned thresholds (beta=1.0) for this best epoch.")

training_args = TrainingArguments(
    output_dir="./tmp_goemotions_7_large_dapt",  # exists, but we DO NOT write checkpoints
    per_device_train_batch_size=cfg.batch_size_train,
    gradient_accumulation_steps=2,
    per_device_eval_batch_size=cfg.batch_size_eval,
    num_train_epochs=cfg.num_epochs_goemotions,
    learning_rate=cfg.lr_goemotions,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    logging_steps=200,
    eval_strategy="epoch",
    save_strategy="no",              # <-- prevents disk writes that were crashing you
    load_best_model_at_end=False,    # <-- no checkpoints
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_go_tok["train"],
    eval_dataset=ds_go_tok["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
)

cb = BestEpochThresholdsCallback(trainer, ds_go_tok["validation"], TARGET_EMOTIONS)
trainer.add_callback(cb)

trainer.train()

if cb.best_thr_go is None:
    raise RuntimeError("No thresholds were captured. Ensure eval_strategy='epoch' and validation split is non-empty.")

print("\n🏁 Phase 1 complete (no checkpoints saved).")
print("Best epoch (by eval_macro_f1 @0.5):", cb.best_epoch, "| eval_macro_f1:", cb.best_macro_f1)

print("\n✅ Tuned thresholds (GoEmotions val, beta=1.0) for BEST epoch:")
for name, t in zip(TARGET_EMOTIONS, cb.best_thr_go):
    print(f"{name:14s}: {t:.3f}")

print("\n===== Sanity Check (GoEmotions val, tuned thresholds @ best epoch) =====")
for name, stats in cb.best_sanity["per_class"].items():
    print(f"{name:15s}  P={stats['p']:.3f}  R={stats['r']:.3f}  F1={stats['f1']:.3f}")
print(f"Macro F1: {cb.best_sanity['macro_f1_tuned']:.3f} | Micro F1: {cb.best_sanity['micro_f1_tuned']:.3f}")

best_thr_go = cb.best_thr_go  # <-- use this later if you want

# ============================================================
# 4. LOAD & PREPARE HIV SYNTHETIC DATA
# ============================================================

print("Building HIV-contextualized component from the HIVEmotions repo...")
hiv_datasets = {}
for split in ["train","validation"]:
    a = authored[split]
    hiv_datasets[split] = Dataset.from_dict({
        "text":   a["text"],
        "labels": [vec(l) for l in a["labels"]],
        "is_synthetic": [bool(int(v)) if str(v).strip() not in ("","None") else False for v in a["is_synthetic"]],
    })
hiv_datasets["test"] = Dataset.from_dict({"text": [], "labels": [], "is_synthetic": []})
print("HIV 7-emotion data: train=%d samples" % len(hiv_datasets["train"]))

hiv_tok = {}
for split, ds in hiv_datasets.items():
    if len(ds) == 0:
        hiv_tok[split] = ds
        continue
    tokenized = ds.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=cfg.max_length,
        ),
        batched=True,
        remove_columns=["text"],
        load_from_cache_file=False,
    )
    tokenized = tokenized.map(
        lambda batch: {"labels": [[float(x) for x in lab] for lab in batch["labels"]]},
        batched=True,
        load_from_cache_file=False,
    )
    tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels", "is_synthetic"])
    hiv_tok[split] = tokenized

print(f"HIV 7-emotion data: train={len(hiv_tok['train'])} samples")

# ============================================================
# 5. MERGE & DATALOADERS
# ============================================================

def add_is_synth_false(ex):
    ex["is_synthetic"] = False
    return ex

ds_go_with_synth = ds_go_tok.map(add_is_synth_false, load_from_cache_file=False)

final_datasets = {}
for split in ["train", "validation", "test"]:
    go_ds = ds_go_with_synth[split]
    hiv_ds = hiv_tok[split]
    final_datasets[split] = concatenate_datasets([go_ds, hiv_ds]) if (len(go_ds) + len(hiv_ds)) > 0 else Dataset.from_dict(
        {"input_ids": [], "attention_mask": [], "labels": [], "is_synthetic": []}
    )

def collate_fn(batch):
    return {
        "input_ids": torch.stack([torch.tensor(x["input_ids"], dtype=torch.long) for x in batch]),
        "attention_mask": torch.stack([torch.tensor(x["attention_mask"], dtype=torch.long) for x in batch]),
        "labels": torch.stack([torch.tensor(x["labels"], dtype=torch.float) for x in batch]),
        "is_synthetic": torch.tensor([x["is_synthetic"] for x in batch], dtype=torch.bool),
    }

train_loader = DataLoader(final_datasets["train"], batch_size=cfg.batch_size_train, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(final_datasets["validation"], batch_size=cfg.batch_size_eval, shuffle=False, collate_fn=collate_fn)

# ============================================================
# 6. FOCAL LOSS & OPTIMIZER
# ============================================================

def compute_pos_weight(dataset):
    labels_list = [ex["labels"].numpy() if hasattr(ex["labels"], "numpy") else np.array(ex["labels"]) for ex in dataset]
    labels = np.stack(labels_list)
    pos = labels.sum(axis=0)
    neg = len(labels) - pos
    return np.where(pos > 0, neg / pos, 1.0).astype(np.float32)

pos_weight = compute_pos_weight(final_datasets["train"])
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float, device=device)

class FocalBCELossWithLogits(nn.Module):
    def __init__(self, pos_weight=None, gamma=2.0, label_smoothing=0.1):
        super().__init__()
        self.register_buffer("pos_weight", pos_weight)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets, is_synthetic):
        if self.label_smoothing > 0:
            targets = targets * (1 - self.label_smoothing) + 0.5 * self.label_smoothing
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pos_weight, reduction="none")
        if self.gamma > 0:
            p = torch.sigmoid(logits)
            pt = p * targets + (1 - p) * (1 - targets)
            bce = (1 - pt).pow(self.gamma) * bce
        weight = torch.where(is_synthetic, cfg.synthetic_weight_factor, 1.0).float().unsqueeze(1)
        return (bce * weight).mean()

criterion = FocalBCELossWithLogits(
    pos_weight=pos_weight_tensor,
    gamma=cfg.focal_gamma,
    label_smoothing=cfg.label_smoothing,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr_hiv, weight_decay=cfg.weight_decay)
num_steps = cfg.num_epochs_hiv * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(cfg.warmup_ratio * num_steps),
    num_steps,
)

# ============================================================
# 7. TRAINING LOOP (Phase 2: HIV fine-tuning)
# ============================================================

def run_eval(loader):
    model.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            out = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            )
            all_logits.append(out.logits.cpu().numpy())
            all_labels.append(batch["labels"].numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = 1 / (1 + np.exp(-logits))
    return probs, labels

def tune_thr_beta(probs, labels, beta=2.0):
    thrs = np.linspace(0.1, 0.9, 17)
    best = np.full(7, 0.5, dtype=np.float32)

    for i in range(7):
        y, p = labels[:, i], probs[:, i]
        best_score = -1.0
        best_t = 0.5
        for t in thrs:
            pred = (p >= t).astype(int)
            tp = ((pred == 1) & (y == 1)).sum()
            fp = ((pred == 1) & (y == 0)).sum()
            fn = ((pred == 0) & (y == 1)).sum()
            if tp + fp == 0 or tp + fn == 0:
                score = 0.0
            else:
                prec = tp / (tp + fp + 1e-8)
                rec = tp / (tp + fn + 1e-8)
                score = (1 + beta**2) * prec * rec / (beta**2 * prec + rec + 1e-8)
            if score > best_score:
                best_score = score
                best_t = float(t)
        best[i] = best_t
    return best

best_metric = -1.0
best_state = None
no_improve = 0

model.train()
for epoch in range(1, cfg.num_epochs_hiv + 1):
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
        )
        loss = criterion(
            out.logits,
            batch["labels"].to(device).float(),
            batch["is_synthetic"].to(device),
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += float(loss.item())

    probs_val, labels_val = run_eval(val_loader)
    thr_recall = tune_thr_beta(probs_val, labels_val, beta=2.0)
    preds = (probs_val >= thr_recall[None, :]).astype(int)
    f1_macro = f1_score(labels_val, preds, average="macro", zero_division=0)

    print(f"Epoch {epoch} | Train Loss: {total_loss/len(train_loader):.4f} | Val Macro-F1: {f1_macro:.4f}")

    if f1_macro > best_metric + 1e-4:
        best_metric = f1_macro
        no_improve = 0
        best_state = {"model": model.state_dict(), "thr_recall": thr_recall, "epoch": epoch}
    else:
        no_improve += 1
        if no_improve >= cfg.patience:
            print("🛑 Early stopping.")
            break

if best_state:
    model.load_state_dict(best_state["model"])
    best_thr_recall = best_state["thr_recall"]
    print(f"✅ Loaded best Phase 2 from epoch {best_state['epoch']} | Val Macro-F1={best_metric:.4f}")
else:
    best_thr_recall = np.full(7, 0.5, dtype=np.float32)

def print_metrics(probs, labels, thrs, label_names, title="Metrics"):
    preds = (probs >= thrs[None, :]).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average=None, zero_division=0)

    print(f"\n===== {title} PER-EMOTION =====")
    for i, name in enumerate(label_names):
        print(f"{name:15s}  P={prec[i]:.3f}  R={rec[i]:.3f}  F1={f1[i]:.3f}")

    macro = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    micro = precision_recall_fscore_support(labels, preds, average="micro", zero_division=0)

    print(f"\n===== {title} GLOBAL =====")
    print(f"Macro Precision: {macro[0]:.3f}")
    print(f"Macro Recall:    {macro[1]:.3f}")
    print(f"Macro F1:        {macro[2]:.3f}")
    print(f"Micro F1:        {micro[2]:.3f}")

print("\n📊 Evaluating final HIV-tuned model on validation set...")
probs_val, labels_val = run_eval(val_loader)
print_metrics(probs_val, labels_val, best_thr_recall, TARGET_EMOTIONS, title="HIV Validation (best checkpoint)")

# ============================================================
# 8. SAVE (OPTIONAL) — comment out if you are still low on disk
# ============================================================

# If you still hit disk errors, comment out the entire block below.
model.save_pretrained(cfg.save_dir)
tokenizer.save_pretrained(cfg.save_dir)
np.save(os.path.join(cfg.save_dir, "best_thr_recall.npy"), best_thr_recall)
with open(os.path.join(cfg.save_dir, "label_names.json"), "w") as f:
    json.dump(TARGET_EMOTIONS, f)
with open(os.path.join(cfg.save_dir, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

print(f"\n🎉 Training complete. Model saved to: {cfg.save_dir}")


## Stage 3 — Deployed F1-oriented thresholds

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def tune_thr_f1(probs, labels, label_names=None):
    """
    Tune a per-label threshold that maximizes F1 for each emotion.
    probs:  np.array of shape [N, num_labels] with probabilities
    labels: np.array of shape [N, num_labels] with 0/1 ground truth
    """
    thrs = np.linspace(0.05, 0.95, 19)  # search grid
    n_labels = labels.shape[1]
    best = np.full(n_labels, 0.5, dtype=np.float32)

    for i in range(n_labels):
        y = labels[:, i]
        p = probs[:, i]

        best_score = -1.0
        for t in thrs:
            pred = (p >= t).astype(int)

            prec, rec, f1, _ = precision_recall_fscore_support(
                y, pred, average="binary", zero_division=0
            )

            if f1 > best_score:
                best_score = f1
                best[i] = t

    return best

def print_metrics(probs, labels, thrs, label_names, title="Metrics"):
    preds = (probs >= thrs[None, :]).astype(int)

    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )

    print(f"\n===== {title} PER-EMOTION =====")
    for i, name in enumerate(label_names):
        print(f"{name:15s}  P={prec[i]:.3f}  R={rec[i]:.3f}  F1={f1[i]:.3f}")

    macro = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    micro = precision_recall_fscore_support(labels, preds, average="micro", zero_division=0)

    print(f"\n===== {title} GLOBAL =====")
    print(f"Macro Precision: {macro[0]:.3f}")
    print(f"Macro Recall:    {macro[1]:.3f}")
    print(f"Macro F1:        {macro[2]:.3f}")
    print(f"Micro F1:        {micro[2]:.3f}")

print("\n📊 Running final evaluation with F1-tuned thresholds...")

# 1. Run model on validation set (no retraining!)
probs_val, labels_val = run_eval(val_loader)   # probs_val, labels_val are np.arrays

# 2. Tune thresholds to maximize F1 per emotion
best_thr_f1 = tune_thr_f1(probs_val, labels_val, TARGET_EMOTIONS)

print("\n🎯 Best F1 thresholds per emotion:")
for name, t in zip(TARGET_EMOTIONS, best_thr_f1):
    print(f"{name:15s}: {t:.3f}")

# 3. Print metrics with these thresholds
print_metrics(
    probs_val,
    labels_val,
    best_thr_f1,
    TARGET_EMOTIONS,
    title="FINAL HIV MODEL (Per-label F1-tuned thresholds)"
)

# Save thresholds as .npy
import os, json
thr_path_npy = os.path.join("thr/m1_7emotion_roberta_large_dapt", "best_thr_f1.npy")
np.save(thr_path_npy, best_thr_f1)
print("💾 Saved thresholds (.npy):", thr_path_npy)

# Save thresholds as JSON (human readable)
thr_dict = {name: float(t) for name, t in zip(TARGET_EMOTIONS, best_thr_f1)}
thr_path_json = os.path.join("thr/m1_7emotion_roberta_large_dapt", "best_thr_f1.json")
with open(thr_path_json, "w") as f:
    json.dump(thr_dict, f, indent=2)
print("💾 Saved thresholds (.json):", thr_path_json)


## Test — backbone alone on the balanced 800-message split
Scored with the paper's standalone convention (the highest-probability emotion among those crossing their thresholds; neutral when none crosses), exactly as in Table 3.2's backbone row and Appendix E.5. The model itself has no neutral output; the deployed pipeline's neutral decisions come from the LLM stage (next cell).

In [ ]:
import torch, numpy as np
from sklearn.metrics import classification_report
def predict_emotions(text, tokenizer, model, thresholds, device):
    enc = tokenizer(
        text, return_tensors="pt", truncation=True,
        padding="max_length", max_length=MAX_LENGTH,
    ).to(device)
    logits = model(**enc).logits.cpu().numpy()[0]
    probs = 1 / (1 + np.exp(-logits))
    binary = (probs >= thresholds).astype(int)

    active = np.where(binary == 1)[0]
    final = "neutral" if len(active) == 0 else TARGET_EMOTIONS[active[np.argmax(probs[active])]]

    return {
        "probs": {e: float(p) for e, p in zip(TARGET_EMOTIONS, probs)},
        "preds": {e: int(b) for e, b in zip(TARGET_EMOTIONS, binary)},
        "final_label": final,
    }


# --------------------------------------------------------------------------- #
# Stage 2 — refinement prompt (FOUR examples)
# --------------------------------------------------------------------------- #


T8 = TARGET_EMOTIONS + ["neutral"]
thr = np.load("thr/m1_7emotion_roberta_large_dapt/best_thr_f1.npy")
texts, gold = list(raw["test"]["text"]), list(raw["test"]["labels"])
preds = [predict_emotions(t, tokenizer, model, thr, device)["final_label"] for t in texts]
print(classification_report(gold, preds, labels=T8, zero_division=0))
print("Paper (this checkpoint): backbone macro P/R/F1 = 0.82 / 0.78 / 0.78")

## Full pipeline (GPT-5-mini refinement) on the test split

In [ ]:
# OPTIONAL — full pipeline (backbone + GPT-5-mini 4-shot) on the test split.
# Needs the OpenAI key in config.py or env. Invalid/failed responses fall back to the backbone label.
import json
from openai import OpenAI
from sklearn.metrics import classification_report
import config

REFINEMENT_MODEL = "gpt-5-mini"

def build_llm_prompt(user_text: str, backbone_output: dict) -> str:
    probs_str = "\n".join(
        f"- {e}: {backbone_output['probs'][e]:.3f}" for e in TARGET_EMOTIONS
    )

    return f"""
You are an expert clinical annotator helping label emotional content in short, user-written messages.
You work with EXACTLY these 8 possible labels:

- anger
- disgust
- embarrassment
- fear
- grief
- nervousness
- sadness
- neutral   (no clear or dominant negative emotion)

You are given:
1) A message written by a person.
2) The output of an emotion classifier (M1), which provides probabilities for the 7 emotions above
   (anger, disgust, embarrassment, fear, grief, nervousness, sadness).
   If none of these emotions clearly fits the message, the most appropriate label is "neutral".

Your task:
- Carefully read the message.
- Use the classifier's probabilities as a hint, but you may OVERRIDE them if they don't match the text.
- Choose EXACTLY ONE final label from the 8 above.
- Briefly justify your choice in 1-3 sentences.
- Be especially careful to distinguish:
  * sadness vs grief (grief is usually linked to loss, death, or bereavement),
  * fear vs nervousness (fear = strong acute danger; nervousness = milder anxiety, worry, anticipation),
  * anger vs disgust (anger = attack/hostility; disgust = moral/physical revulsion),
  * embarrassment (social awkwardness, shame, wanting to hide) vs the others.
- "neutral" means the message is mostly informational or very mild in emotion.

Always answer in VALID JSON, with this exact structure (no extra keys, no extra text):

{{
  "final_label": "<one of: anger, disgust, embarrassment, fear, grief, nervousness, sadness, neutral>",
  "explanation": "<your short explanation>"
}}

--------------------------
FEW-SHOT EXAMPLES
--------------------------

Example 1
Message:
\"\"\"I've been crying all day since I got my HIV diagnosis. I feel completely empty and I don't see how I'll ever be happy again.\"\"\"

M1 emotion probabilities:
- anger: 0.041
- disgust: 0.018
- embarrassment: 0.072
- fear: 0.452
- grief: 0.731
- nervousness: 0.311
- sadness: 0.845

Assistant JSON answer:
{{
  "final_label": "sadness",
  "explanation": "The message expresses intense sadness and despair about the diagnosis. Although there are elements of fear and grief, the dominant tone is pervasive sadness and hopelessness."
}}

--------------------------

Example 2
Message:
\"\"\"People act like I'm dirty just because I'm HIV positive. I'm so sick of their judgment, it makes me want to scream.\"\"\"

M1 emotion probabilities:
- anger: 0.812
- disgust: 0.644
- embarrassment: 0.221
- fear: 0.093
- grief: 0.051
- nervousness: 0.102
- sadness: 0.303

Assistant JSON answer:
{{
  "final_label": "anger",
  "explanation": "The speaker is reacting to unfair treatment and judgment with a strong desire to 'scream', which indicates anger more than disgust or sadness."
}}

--------------------------

Example 3
Message:
\"\"\"My HIV test results come back tomorrow and my stomach has been in knots all day. I can't stop thinking about it.\"\"\"

M1 emotion probabilities:
- anger: 0.035
- disgust: 0.010
- embarrassment: 0.047
- fear: 0.521
- grief: 0.022
- nervousness: 0.781
- sadness: 0.166

Assistant JSON answer:
{{
  "final_label": "nervousness",
  "explanation": "The message focuses on anticipation, worry, and physical signs of anxiety about future results, which fits nervousness better than full-blown fear."
}}

--------------------------

Example 4
Message:
\"\"\"I have an appointment with my doctor next week to adjust my HIV medication. They said we'll review my latest lab results.\"\"\"

M1 emotion probabilities:
- anger: 0.012
- disgust: 0.008
- embarrassment: 0.014
- fear: 0.042
- grief: 0.003
- nervousness: 0.089
- sadness: 0.054

Assistant JSON answer:
{{
  "final_label": "neutral",
  "explanation": "The message is mainly factual and logistical about an upcoming appointment, without a clear negative emotional tone."
}}

--------------------------
NOW ANALYZE THIS NEW MESSAGE
--------------------------

Message:
\"\"\"{user_text}\"\"\"

M1 emotion probabilities:
{probs_str}

Remember:
- You must choose exactly ONE label from: anger, disgust, embarrassment, fear, grief, nervousness, sadness, neutral.
- Use M1 probabilities as a hint, but override them if the text suggests a different emotion.
- Respond ONLY with valid JSON in the specified format.

Assistant JSON answer:
""".strip()

def call_llm(client, prompt: str) -> str:
    """gpt-5-mini rejects the temperature parameter, so it is not sent."""
    try:
        resp = client.chat.completions.create(
            model=REFINEMENT_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "You are an assistant that ALWAYS responds with a single JSON "
                        "object. This message explicitly contains the word json to "
                        "satisfy the API requirement."
                    ),
                },
                {"role": "user", "content": prompt},
            ],
            response_format={"type": "json_object"},
        )
        return resp.choices[0].message.content
    except Exception as exc:
        return json.dumps({"final_label": None, "explanation": f"call failed: {exc}"})


# --------------------------------------------------------------------------- #

client = OpenAI(api_key=config.get_openai_key())
valid = set(TARGET_EMOTIONS) | {"neutral"}
final = []
for t in texts:
    bo = predict_emotions(t, tokenizer, model, thr, device)
    raw = call_llm(client, build_llm_prompt(t, bo))
    try:
        label = json.loads(raw).get("final_label")
    except Exception:
        label = None
    final.append(label if label in valid else None)
pairs = [(g, l) for g, l in zip(gold, final) if l is not None]
if len(pairs) < len(final):
    print(f"WARNING: {len(final)-len(pairs)} failed LLM call(s) excluded from the report — rerun for exact comparability.")
print(classification_report([g for g,_ in pairs], [l for _,l in pairs], labels=T8, zero_division=0))
print("Paper (this checkpoint): pipeline macro P/R/F1 = 0.90 / 0.88 / 0.88, micro 0.88")